# Spec-FastGS — Mip-NeRF 360 Multi-Resolution Sweep & Upload Pipeline

> **Kernel**: `thesis_env` (Set via Kernel → Change Kernel after running `bosch_setup_thesis.ipynb` once to register it)

Runs the full Spec-FastGS training sweep on Mip-NeRF 360 scenes inside the BOSCH server environment across multiple selected image resolutions (`images`, `images_2`, `images_4`, `images_8`), automatically archiving, uploading results to Hugging Face, and purging local output files to save disk space.

**What this notebook does:**
1. **Proxy & Env Check**: Sets up environment variables for the BOSCH server.
2. **Target Resolutions Selection**: Choose any combination of resolutions (`["images", "images_2", "images_4", "images_8"]`).
3. **Resolution & Layout Validation**: Verifies selected resolution folders exist for all 9 scenes (`bicycle`, `flowers`, `garden`, `stump`, `treehill`, `room`, `counter`, `kitchen`, `bonsai`).
4. **Sweep Execution**: Invokes `run_mip360.sh` for each selected resolution with `DATA_ROOT="/home/ghp4hc/datasets/datasets/mipneft360"`.
5. **Results Summary**: Formats and prints quantitative metrics (`results_grouped.json`) in a neat table for each resolution.
6. **Archiving & HF Upload**: Zips and uploads each resolution's output (`images.zip`, `images2.zip`, `images4.zip`, `images8.zip`) directly to `DiBiay/spec-fastgs-mipnerf360-result`.
7. **Auto Cleanup**: Purges the local zip archive and output directory after a successful upload to save BOSCH server disk space.

## c00 — Proxy Settings
Sets the BOSCH proxy for external connectivity.

In [ ]:
# ── Proxy (required for HF / huggingface cache / diagnostic endpoints) ────────
import os

PROXY = 'http://rb-proxy-sl.bosch.com:8080'
HOME  = os.path.expanduser('~')

os.environ['http_proxy']  = PROXY
os.environ['https_proxy'] = PROXY
os.environ['HTTP_PROXY']  = PROXY
os.environ['HTTPS_PROXY'] = PROXY

print(f'Proxy set to: {PROXY}')

## c01 — Config, Target Resolutions & Kernel Check
Defines paths, target resolution list (`["images", "images_2", "images_4", "images_8"]`), and double-checks if the correct virtual environment kernel is loaded.

In [ ]:
# ── Configurations & environment variables check ─────────────────────────────
import os
import sys

HOME = os.path.expanduser('~')

# Select resolutions to iterate through: 'images', 'images_2', 'images_4', 'images_8'
TARGET_RESOLUTIONS = ["images", "images_2", "images_4", "images_8"]

# Robustly resolve repository root path
if os.path.isdir('/home/ghp4hc/thesis-all/spec-fastgs'):
    REPO_ROOT = '/home/ghp4hc/thesis-all/spec-fastgs'
elif os.path.isdir(os.path.join(os.getcwd(), 'thesis-all', 'spec-fastgs')):
    REPO_ROOT = os.path.join(os.getcwd(), 'thesis-all', 'spec-fastgs')
else:
    REPO_ROOT = os.path.join(os.getcwd(), 'spec-fastgs')

ENV_NAME = 'thesis_env'

print(f'Active Python        : {sys.executable}')
print(f'Active Kernel name   : {ENV_NAME}')
print(f'Repository Root      : {REPO_ROOT}')
print(f'Target Resolutions   : {TARGET_RESOLUTIONS}')

assert REPO_ROOT in sys.executable or ENV_NAME in sys.executable or '.conda' in sys.executable, \
    f"WARNING: You are not running on the '{ENV_NAME}' kernel! Please select Kernel -> Change Kernel -> Python ({ENV_NAME})"

## c02 — Imports & GPU Validation
Verifies hardware detection and compiled custom modules availability.

In [ ]:
# ── Verification of PyTorch & custom submodules ────────────────────────────────
import torch
print('PyTorch version :', torch.__version__)
print('CUDA Available  :', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU device name :', torch.cuda.get_device_name(0))
    print('Compute Cap.    :', torch.cuda.get_device_capability(0))

import diff_gaussian_rasterization_fastgs
import simple_knn
import fused_ssim
print('rasterizer      : OK')
print('simple-knn      : OK')
print('fused-ssim      : OK')

try:
    import huggingface_hub
    print('huggingface_hub : OK')
except ImportError:
    print('huggingface_hub : MISSING (will auto-install during the upload step)')

## c03 — Verify Dataset Path
Checks that the Mip-NeRF 360 source dataset is available on the server.

In [ ]:
# ── Verify Dataset Directory ──────────────────────────────────────────────────
import os
import sys

src_root = "/home/ghp4hc/datasets/datasets/mipneft360"
assert os.path.exists(src_root), f"Dataset path not found at {src_root}! Check that the datasets are downloaded."
print(f"✅ Found dataset source root: {src_root}")

print(f"\n📂 Source datasets directory content:")
print(os.listdir(src_root))

## c04 — Verify Target Resolutions Layout
Validates all 9 scenes for each resolution in `TARGET_RESOLUTIONS`.

In [ ]:
# ── Verify Mip-NeRF 360 Dataset Layout for all Target Resolutions ────────────
import os

src_root = "/home/ghp4hc/datasets/datasets/mipneft360"
MIP360_SCENES = [
    "bicycle", "flowers", "garden", "stump", "treehill",
    "room", "counter", "kitchen", "bonsai",
]

for resolution in TARGET_RESOLUTIONS:
    print(f"\n🔍 Verifying resolution '{resolution}' under {src_root} ...")
    missing = []
    for scene in MIP360_SCENES:
        v2_path = os.path.join(src_root, "360_v2", scene)
        extra_path = os.path.join(src_root, "360_extra_scenes", scene)
        
        if os.path.isdir(v2_path):
            scene_dir = v2_path
        elif os.path.isdir(extra_path):
            scene_dir = extra_path
        else:
            scene_dir = None
            
        if scene_dir is None:
            status = "MISSING (scene folder not found)"
            missing.append(scene)
        else:
            images_dir = os.path.join(scene_dir, resolution)
            if not os.path.isdir(images_dir):
                status = f"MISSING ({resolution} not found; has: {sorted(os.listdir(scene_dir))[:6]})"
                missing.append(scene)
            else:
                n_imgs = len([f for f in os.listdir(images_dir) if f.lower().endswith(('.jpg', '.jpeg', '.png'))])
                status = f"OK ({n_imgs} images)"
                
        print(f"  {scene:<12s} {status}")

    if missing:
        print(f"⚠️  {len(missing)}/{len(MIP360_SCENES)} scene(s) missing {resolution}: {missing}")
    else:
        print(f"✅ All {len(MIP360_SCENES)} scenes verified for '{resolution}'.")

## c05 — Run Spec-FastGS Sweeps, Upload & Auto-Cleanup
Iterates through each resolution in `TARGET_RESOLUTIONS`, runs `run_mip360.sh`, summarizes metrics, zips output, uploads to Hugging Face, and purges local zip & output folder to save disk space.

In [ ]:
# ── Prepare script inputs & HF API ───────────────────────────────────────────
import subprocess
import os
import sys
import json
import shutil

try:
    from huggingface_hub import HfApi
except ImportError:
    print("Installing huggingface_hub via pip...")
    PROXY = 'http://rb-proxy-sl.bosch.com:8080'
    subprocess.run([sys.executable, "-m", "pip", "install", "huggingface_hub", "--proxy", PROXY], check=True)
    from huggingface_hub import HfApi

# Robustly resolve repository root path
if os.path.isdir('/home/ghp4hc/thesis-all/spec-fastgs'):
    REPO_ROOT = '/home/ghp4hc/thesis-all/spec-fastgs'
elif os.path.isdir(os.path.join(os.getcwd(), 'thesis-all', 'spec-fastgs')):
    REPO_ROOT = os.path.join(os.getcwd(), 'thesis-all', 'spec-fastgs')
else:
    REPO_ROOT = os.path.join(os.getcwd(), 'spec-fastgs')

DATA_ROOT = "/home/ghp4hc/datasets/datasets/mipneft360"
HF_TOKEN = os.environ.get('HF_TOKEN', '<YOUR_HF_TOKEN>')
HF_REPO = os.environ.get('HF_REPO', 'DiBiay/spec-fastgs-mipnerf360-result')

venv_bin = os.path.dirname(sys.executable)
cuda_home = os.environ.get('CUDA_HOME', '')
if not cuda_home:
    search_script = 'which nvcc 2>/dev/null || (for init in /etc/profile /etc/profile.d/modules.sh; do [ -f "$init" ] && source "$init"; done && for mod in cuda/11.7 cuda/11.8 cuda/12.1 cuda/12.6; do module load "$mod" 2>/dev/null; done && which nvcc 2>/dev/null)'
    r_nvcc = subprocess.run(['bash', '-c', search_script], capture_output=True, text=True)
    if r_nvcc.returncode == 0 and r_nvcc.stdout.strip():
        cuda_home = os.path.dirname(os.path.dirname(r_nvcc.stdout.strip()))
    else:
        cuda_home = '/usr/local/cuda'

# Map resolution to target filename in Hugging Face repository
zip_name_map = {
    'images':   'images.zip',
    'images_2': 'images2.zip',
    'images_4': 'images4.zip',
    'images_8': 'images8.zip',
}

MIP360_SCENES = [
    "bicycle", "flowers", "garden", "stump", "treehill",
    "room", "counter", "kitchen", "bonsai",
]

def fmt(x, nd=4):
    return f"{x:.{nd}f}" if isinstance(x, (int, float)) else "-"

for resolution in TARGET_RESOLUTIONS:
    print(f"\n========================================================================")
    print(f" 🚀 STARTING SPEC-FASTGS SWEEP FOR RESOLUTION: {resolution}")
    print(f"========================================================================")
    
    logfile = os.path.join(os.path.dirname(REPO_ROOT), f"mip360_{resolution}_specfastgs_run.log")
    target_hf_filename = zip_name_map.get(resolution, f"{resolution}.zip")
    
    # 1. Run bash sweep
    cmd = f'''
    export PATH={venv_bin}:{cuda_home}/bin:$PATH
    export LD_LIBRARY_PATH={cuda_home}/lib64:$LD_LIBRARY_PATH
    export CUDA_VISIBLE_DEVICES=0
    export DATA_ROOT={DATA_ROOT}
    export IMAGES={resolution}
    cd "{REPO_ROOT}"
    bash run_mip360.sh > "{logfile}" 2>&1
    '''
    r = subprocess.run(['bash', '-c', cmd])
    
    print(f"--- tail of {logfile} ---")
    if os.path.exists(logfile):
        with open(logfile, 'r') as f:
            lines = f.readlines()
            print(''.join(lines[-40:]))
            
    # 2. Print quantitative summary
    out_root = os.path.join(REPO_ROOT, "output", f"mip360_{resolution}")
    print(f"\n📊 Quantitative Results Summary for '{resolution}':")
    header = f"{'scene':<12s}{'PSNR':>8s}{'SSIM':>8s}{'LPIPS':>8s}{'Spec_PSNR':>11s}{'ASG_IoU':>9s}{'#Gauss':>10s}{'time':>10s}"
    print(header)
    print("-" * len(header))
    for scene in MIP360_SCENES:
        out_dir = os.path.join(out_root, scene)
        results_path = os.path.join(out_dir, "results_grouped.json")
        info_path = os.path.join(out_dir, "train_info.json")
        if not os.path.exists(results_path):
            print(f"{scene:<12s}  (no results_grouped.json)")
            continue
        with open(results_path) as f:
            res = json.load(f)
        sr = next(iter(res.values()))
        rr = next(iter(sr.values()))
        main = rr.get("main_metrics", {})
        aux = rr.get("aux_metrics", {})
        info = {}
        if os.path.exists(info_path):
            with open(info_path) as f:
                info = json.load(f)
        print(f"{scene:<12s}{fmt(main.get('PSNR')):>8s}{fmt(main.get('SSIM')):>8s}{fmt(main.get('LPIPS')):>8s}"
              f"{fmt(aux.get('Spec_PSNR')):>11s}{fmt(aux.get('ASG_Residual_IoU')):>9s}"
              f"{str(info.get('final_gaussians', '-')):>10s}{str(info.get('training_time_formatted', '-')):>10s}")
              
    # 3. Archive results into zip
    zip_out = os.path.join(os.path.dirname(REPO_ROOT), f"spec_fastgs_output_mip360_{resolution}")
    archive_file = zip_out + ".zip"
    if os.path.isdir(out_root):
        print(f"\n📦 Archiving '{out_root}' -> '{archive_file}' ...")
        if os.path.exists(archive_file):
            os.remove(archive_file)
        shutil.make_archive(zip_out, "zip", out_root)
        print("Archived size:", round(os.path.getsize(archive_file) / 1e6, 1), "MB")
        
        # 4. Upload to Hugging Face
        if HF_TOKEN and HF_TOKEN != '<YOUR_HF_TOKEN>':
            print(f"📤 Uploading '{archive_file}' as '{target_hf_filename}' to Hugging Face repo '{HF_REPO}'...")
            try:
                api = HfApi()
                try:
                    api.repo_info(repo_id=HF_REPO, repo_type="dataset", token=HF_TOKEN)
                except Exception:
                    api.create_repo(repo_id=HF_REPO, repo_type="dataset", token=HF_TOKEN, private=True)
                    
                api.upload_file(
                    path_or_fileobj=archive_file,
                    path_in_repo=target_hf_filename,
                    repo_id=HF_REPO,
                    repo_type="dataset",
                    token=HF_TOKEN,
                )
                print(f"🎉 [SUCCESS] Uploaded '{target_hf_filename}' to '{HF_REPO}'!")
                
                # 5. Purge local zip and output directory to prevent OOM & save disk space
                print(f"🗑️ Cleaning up local zip file '{archive_file}' and output directory '{out_root}'...")
                if os.path.exists(archive_file):
                    os.remove(archive_file)
                if os.path.isdir(out_root):
                    shutil.rmtree(out_root, ignore_errors=True)
                print(f"✅ Local disk space freed for resolution '{resolution}'!")
            except Exception as e:
                print(f"❌ [ERROR] HF upload failed for {resolution}: {e}")
                print(f"⚠️  Retaining local zip '{archive_file}' and output folder '{out_root}' for inspection.")
        else:
            print(f"⚠️  Skipping HF upload (HF_TOKEN not set or default placeholder used).")
    else:
        print(f"❌ [ERROR] Output directory '{out_root}' not found. Skipping zip/upload for {resolution}.")